# UNSW-NB15: контроль порту і змісту CTI при фіксованому бюджеті

Це ті самі контролі, що й на CIC-IDS2017 (стратифікований спліт), але на чинному хронологічному спліті UNSW з статті (quantile 0.70/0.85 по Stime). Усі моделі навчаються на 500 деревах без early stopping.

## Гіпотези і критерії (записано ДО запуску)
- **(c) Порт.** Чи дає CTI щось понад сирий номер порту призначення (`dsport`, виключений з F0 за протоколом)?
  - Критерій: при 500 деревах ΔAP(F1 − F0+port) > 0, і нижня межа 95% блочного bootstrap CI (блоки = хвилини Stime) > 0.
  - Додатково звітується ΔAP(F0+port − F0).
- **(d) Зміст CTI.** Чи важать справжні значення CVE-ознак, а не лише групування за сервісом?
  - Модель F0 + числові CTI-ознаки (справжні) порівнюється з F0 + ті самі ознаки, переставлені між сервісами (групування збережене, значення чужого сервісу).
  - Критерій: ΔAP(real − permuted) > 0, і нижня межа CI > 0.
- Обидва результати звітуються незалежно від знаку. Тестів два, тому висновок робиться лише тоді, коли CI явно відокремлений від нуля.

Примітка: у UNSW F1 немає EPSS-оцінок. `n_epss`, `service_max_epss` і `service_in_kev` виключені як константи. Числові CTI-ознаки тут такі: pool_size, epss_coverage, n_generic_os_matches, n_specific_product_matches.

In [1]:
import pandas as pd, numpy as np, lightgbm as lgb
from sklearn.metrics import average_precision_score

PATH = "/srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet"
TS_COL = "Stime"
SEED, ROUNDS, N_BOOT = 42, 500, 500

F0_UNSW = ['state','dur','sbytes','dbytes','sttl','dttl','sloss','dloss','Sload','Dload','Spkts','Dpkts',
'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Sintpkt',
'Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login',
'ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
'ct_dst_src_ltm','proto','service']
NUM_CTX = ['cve_ctx_pool_size','cve_ctx_epss_coverage','cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']
F1_UNSW = F0_UNSW + ['cve_ctx_service'] + NUM_CTX
CAT_COLS = {"state", "proto", "service", "cve_ctx_service"}

df = pd.read_parquet(PATH)
df.columns = [c.strip() for c in df.columns]
PORT_COL = next((c for c in ["dsport", "dst_port", "destination_port"] if c in df.columns), None)
print("rows:", len(df), "| PORT_COL:", PORT_COL)
print("відсутні колонки:", [c for c in F1_UNSW + ["Label", TS_COL] if c not in df.columns] or "— жодної")

df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")
y_np = pd.to_numeric(df["Label"], errors="coerce").astype(int).to_numpy()

rows: 2059418 | PORT_COL: dsport
відсутні колонки: — жодної


In [2]:
# 1. Порт як число: в UNSW dsport буває десятковим, шістнадцятковим ('0x...') або '-'
def parse_port(v):
    s = str(v).strip().lower()
    if s.startswith("0x"):
        try: return float(int(s, 16))
        except ValueError: return np.nan
    try: return float(s)
    except ValueError: return np.nan

raw_port = df[PORT_COL].astype(str)
uniq = raw_port.unique()
port_map = {u: parse_port(u) for u in uniq}
df["_port_num"] = raw_port.map(port_map).astype(float)
print("унікальних значень порту:", len(uniq), "| не розпізнано (NaN):", int(df["_port_num"].isna().sum()))
print("приклади нерозпізнаних:", [u for u in uniq if np.isnan(port_map[u])][:10])

унікальних значень порту: 64630 | не розпізнано (NaN): 7
приклади нерозпізнаних: ['-']


In [3]:
# 2. Спліт як у статті + перевірка, що числові CTI-ознаки є константами всередині сервісу
t = pd.to_numeric(df[TS_COL])
b1 = t.quantile(0.70, interpolation="nearest"); b2 = t.quantile(0.85, interpolation="nearest")
tr = (t < b1).to_numpy(); va = ((t >= b1) & (t < b2)).to_numpy(); te = (t >= b2).to_numpy()
print("розміри:", int(tr.sum()), int(va.sum()), int(te.sum()), "  (у статті: 1,441,568 / 308,915 / 308,935)")
print("AP випадкового класифікатора на test:", round(float(y_np[te].mean()), 6))

key = df["cve_ctx_service"].astype(str)
print("\nМакс. к-сть різних значень кожної числової CTI-ознаки в межах сервісу (1 = константа):")
print(df.groupby(key, observed=True)[NUM_CTX].nunique(dropna=False).max().to_string())
print("\nк-сть сервісів у cve_ctx_service:", key.nunique())

розміри: 1441568 308915 308935   (у статті: 1,441,568 / 308,915 / 308,935)
AP випадкового класифікатора на test: 0.082069

Макс. к-сть різних значень кожної числової CTI-ознаки в межах сервісу (1 = константа):
cve_ctx_pool_size                     1
cve_ctx_epss_coverage                 1
cve_ctx_n_generic_os_matches          1
cve_ctx_n_specific_product_matches    1

к-сть сервісів у cve_ctx_service: 28


In [4]:
# 3. Моделі, 500 дерев кожна, однакові заморожені гіперпараметри
SPW = (y_np[tr] == 0).sum() / y_np[tr].sum()
PARAMS = dict(objective="binary", metric="None", seed=SEED, num_leaves=15, learning_rate=0.05,
              min_data_in_leaf=50, verbosity=-1, scale_pos_weight=SPW)
print("scale_pos_weight:", round(float(SPW), 4), "  (у статті: 28.2135)")

def fit_fixed(src, cols, tag):
    frame = src[cols].copy()
    cats = [c for c in CAT_COLS if c in cols]
    for c in cats:
        frame[c] = pd.Categorical(frame[c], categories=pd.Categorical(frame.loc[tr, c]).categories)
    m = lgb.train(PARAMS, lgb.Dataset(frame[tr], label=y_np[tr], categorical_feature=cats), num_boost_round=ROUNDS)
    print(f"[{tag}] навчено {ROUNDS} дерев, {len(cols)} ознак")
    return m, frame

# переставлені між сервісами числові CTI-ознаки
svc_stats = df.groupby(key, observed=True)[NUM_CTX].first()
svc_stats = svc_stats[~svc_stats.index.isin(["nan", "<NA>", "None"])]
perm = np.random.default_rng(SEED).permutation(len(svc_stats))
df_perm = df.copy()
for c in NUM_CTX:
    mapping = dict(zip(svc_stats.index, svc_stats[c].to_numpy()[perm]))
    df_perm[c] = pd.to_numeric(key.map(mapping).where(key.isin(list(mapping.keys())), df[c]), errors="coerce")
print("сервісів переставлено:", len(svc_stats))

models = {
    "F0":          fit_fixed(df, F0_UNSW, "F0"),
    "F1":          fit_fixed(df, F1_UNSW, "F1"),
    "F0+port":     fit_fixed(df, F0_UNSW + ["_port_num"], "F0 + dsport"),
    "F0+num_real": fit_fixed(df, F0_UNSW + NUM_CTX, "F0 + числові CTI (справжні)"),
    "F0+num_perm": fit_fixed(df_perm, F0_UNSW + NUM_CTX, "F0 + числові CTI (переставлені між сервісами)"),
}

scale_pos_weight: 28.2135   (у статті: 28.2135)
сервісів переставлено: 28
[F0] навчено 500 дерев, 41 ознак
[F1] навчено 500 дерев, 46 ознак
[F0 + dsport] навчено 500 дерев, 42 ознак
[F0 + числові CTI (справжні)] навчено 500 дерев, 45 ознак
[F0 + числові CTI (переставлені між сервісами)] навчено 500 дерев, 45 ознак


In [5]:
# 4. Криві AP при однаковому бюджеті (довідка)
rows = []
for r in [50, 100, 200, 290, 500]:
    a = {k: average_precision_score(y_np[te], m.predict(f[te], num_iteration=r)) for k, (m, f) in models.items()}
    rows.append({"rounds": r, **{f"AP_{k}": v for k, v in a.items()}})
    print(f"rounds={r:>3}  " + "  ".join(f"{k}={v:.6f}" for k, v in a.items()))
curve = pd.DataFrame(rows)

rounds= 50  F0=0.971914  F1=0.977164  F0+port=0.978799  F0+num_real=0.972230  F0+num_perm=0.975133
rounds=100  F0=0.977313  F1=0.980958  F0+port=0.983333  F0+num_real=0.977933  F0+num_perm=0.978756
rounds=200  F0=0.981662  F1=0.983877  F0+port=0.985318  F0+num_real=0.983059  F0+num_perm=0.982827
rounds=290  F0=0.981978  F1=0.983969  F0+port=0.985190  F0+num_real=0.983419  F0+num_perm=0.983323
rounds=500  F0=0.982439  F1=0.983854  F0+port=0.985116  F0+num_real=0.983505  F0+num_perm=0.983265


In [6]:
# 5. Блочний bootstrap CI при 500 деревах (блоки = хвилини Stime)
p = {k: m.predict(f[te], num_iteration=ROUNDS) for k, (m, f) in models.items()}
y_te = y_np[te]
minute = (t[te].to_numpy() // 60).astype(np.int64)
blocks = list(pd.Series(np.arange(len(minute))).groupby(minute).indices.values())
print("рядків у test:", len(y_te), "| хвилинних блоків:", len(blocks))
rng = np.random.default_rng(SEED)

def block_ci(pa, pb):
    point = average_precision_score(y_te, pa) - average_precision_score(y_te, pb)
    vals = []
    for _ in range(N_BOOT):
        j = np.concatenate([blocks[k] for k in rng.integers(0, len(blocks), len(blocks))])
        vals.append(average_precision_score(y_te[j], pa[j]) - average_precision_score(y_te[j], pb[j]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return point, lo, hi, float(np.mean(np.array(vals) > 0))

res = []
for name, a, b in [("F1 − F0 (довідка)", "F1", "F0"),
                   ("(c) F0+port − F0", "F0+port", "F0"),
                   ("(c) F1 − F0+port", "F1", "F0+port"),
                   ("(d) num_real − num_perm", "F0+num_real", "F0+num_perm")]:
    pt, lo, hi, share = block_ci(p[a], p[b])
    res.append({"comparison": name, "dAP": pt, "CI95_lo": lo, "CI95_hi": hi, "P(dAP>0)": share})
    print(f"{name:<26} dAP={pt:+.6f}  95% block-CI [{lo:+.6f}, {hi:+.6f}]  частка реплік >0: {share:.3f}")
ci_df = pd.DataFrame(res)

c_row = ci_df[ci_df.comparison == "(c) F1 − F0+port"].iloc[0]
d_row = ci_df[ci_df.comparison == "(d) num_real − num_perm"].iloc[0]
print("\n(c) CTI дає більше, ніж порт:", "ТАК" if c_row.CI95_lo > 0 else "НІ")
print("(d) Справжні значення CVE важать понад групування за сервісом:", "ТАК" if d_row.CI95_lo > 0 else "НІ")

рядків у test: 308935 | хвилинних блоків: 200
F1 − F0 (довідка)          dAP=+0.001415  95% block-CI [+0.000841, +0.002120]  частка реплік >0: 1.000
(c) F0+port − F0           dAP=+0.002678  95% block-CI [+0.001733, +0.003812]  частка реплік >0: 1.000
(c) F1 − F0+port           dAP=-0.001262  95% block-CI [-0.002097, -0.000625]  частка реплік >0: 0.000
(d) num_real − num_perm    dAP=+0.000240  95% block-CI [-0.000333, +0.001027]  частка реплік >0: 0.742

(c) CTI дає більше, ніж порт: НІ
(d) Справжні значення CVE важать понад групування за сервісом: НІ


In [7]:
# 6. Збереження
out = "/srv/ids_research/data"
curve.to_csv(f"{out}/unsw_controls_curve.csv", index=False)
ci_df.to_csv(f"{out}/unsw_controls_bootstrap_ci.csv", index=False)
print("Збережено: unsw_controls_curve.csv, unsw_controls_bootstrap_ci.csv")

Збережено: unsw_controls_curve.csv, unsw_controls_bootstrap_ci.csv
